In [1]:
import laspy
import numpy as np
import CSF
import pickle
import open3d as o3d
from scipy.interpolate import griddata
from scipy import ndimage as ndi
from pathlib import Path
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import pandas as pd
import matplotlib.pyplot as plt 
from scipy.spatial import ConvexHull
from scipy.spatial import Delaunay
import alphashape
from scipy.interpolate import NearestNDInterpolator
import torch

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def read_las(ruta_las):
    """
    Reads a LAS file and returns the point cloud as a NumPy array.

    :param ruta_las: Path to the LAS file.
    :return: NumPy array of shape (N, 3) containing the point cloud coordinates.
    """
    las = laspy.read(ruta_las)
    points = np.vstack((las.x, las.y, las.z)).transpose()
    return points



In [3]:

def segment_terrain_points(points, resolution = 0.5, altura_min_arbol = 4.0):
    """
    Receives a point cloud and segments terrain points using CSF, then normalizes heights to create a CHM.
    """
    # 1. MDT y NORMALIZACIÓN DE ALTURA
    min_x, max_x = np.min(points[:, 0]), np.max(points[:, 0])
    min_y, max_y = np.min(points[:, 1]), np.max(points[:, 1])

    # Crear grid asegurando que cubra todo
    grid_x, grid_y = np.mgrid[min_x:max_x:resolution, min_y:max_y:resolution]

    csf = CSF.CSF()
    csf.params.bSloopSmooth = True
    csf.params.rigidness = 1
    csf.params.cloth_resolution = 0.75
    csf.params.time_step = 0.75
    csf.params.iterations = 500
    csf.setPointCloud(points)
    ground_indices = CSF.VecInt(); non_ground_indices = CSF.VecInt()
    csf.do_filtering(ground_indices, non_ground_indices)

    puntos_suelo_csf = points[ground_indices]

    mdt_interpolado = griddata(
        points=puntos_suelo_csf[:, :2],
        values=puntos_suelo_csf[:, 2],
        xi=(grid_x, grid_y),
        method='linear',
    )

    if np.isnan(mdt_interpolado).any():
        mask_nan = np.isnan(mdt_interpolado)
        rellenador = NearestNDInterpolator(puntos_suelo_csf[:, :2], puntos_suelo_csf[:, 2])
        coords_nan = np.vstack((grid_x[mask_nan], grid_y[mask_nan])).T
        mdt_interpolado[mask_nan] = rellenador(coords_nan)

    # Nota: Guardar terrain_points es opcional, lo dejo comentado para limpieza
    terrain_points = np.vstack((grid_x.ravel(), grid_y.ravel(), mdt_interpolado.ravel())).T
    np.save("terrain_points.npy", terrain_points)

    # Índices seguros
    idx_x_all = np.clip(((points[:, 0] - min_x) / resolution).astype(int), 0, mdt_interpolado.shape[0] - 1)
    idx_y_all = np.clip(((points[:, 1] - min_y) / resolution).astype(int), 0, mdt_interpolado.shape[1] - 1)

    z_suelo_puntos = mdt_interpolado[idx_x_all, idx_y_all]
    altura_sobre_suelo = points[:, 2] - z_suelo_puntos

    # mask_candidatos = altura_sobre_suelo > altura_min_arbol
    # puntos_arbol_candidatos = points[mask_candidatos]
    # alturas_candidatos = altura_sobre_suelo[mask_candidatos]
    # puntos_resto = points[~mask_candidatos]

    return z_suelo_puntos, altura_sobre_suelo, idx_x_all, idx_y_all, terrain_points

In [4]:
import numpy as np
import alphashape
import math

def calcular_diametro_equivalente_alpha(puntos_arbol, alpha=0.15):
    """
    Calcula el CD (Diámetro de Copa) usando el área de un Alpha Shape 2D.
    
    Args:
        puntos_arbol (np.array): Array de (N, 3) con coordenadas X, Y, Z.
        alpha (float): Nivel de ajuste (0.0 = Convex Hull).
        
    Returns:
        float: Diámetro equivalente (CD).
    """
    # 1. Proyectamos a 2D (solo nos interesan X e Y para la copa)
    puntos_2d = puntos_arbol[:, :2]

    try:
        # 2. Generamos el Alpha Shape
        # El resultado suele ser un Polygon o MultiPolygon de la librería shapely
        hull = alphashape.alphashape(puntos_2d, alpha)

        # 3. Validamos la geometría resultante
        if hull.is_empty:
            # Si el alpha es muy alto para la densidad de puntos, probamos con Convex Hull
            hull = alphashape.alphashape(puntos_2d, 0.0)
        
        # Obtenemos el área (en metros cuadrados)
        area_2d = hull.area

        # 4. Aplicamos la fórmula del Diámetro Equivalente. Basada en el área de un círculo: A = pi * (CD/2)^2 => CD = 2 * sqrt(A / pi)
        # CD = 2 * sqrt(Area / pi)
        cd_equivalente = 2 * math.sqrt(area_2d / math.pi)

        return cd_equivalente, area_2d

    except Exception as e:
        print(f"Error en el cálculo: {e}")
        return 0.0, 0.0

In [6]:
import numpy as np

def calcular_agb(h, cd):
    """
    Calcula la Biomasa Aérea Total (AGB) en Megagramos (Mg).
    
    Args:
        h (float): Altura total del árbol en metros.
        cd (float): Diámetro de copa equivalente en metros.
    
    Returns:
        float: Biomasa estimada en Mg (toneladas de peso seco).
    """
    
    # 1. Parámetros base y de corrección (sigma_v = 0.69)
    alpha_base = 0.016
    beta_base = 2.013
    sigma_v = 0.69
    cf = np.exp((sigma_v**2) / 2) # Factor de corrección logarítmica (~1.2687)

    alpha_g = 0.093
    beta_g = -0.223

    # 3. Cálculo de los coeficientes finales
    alpha_total = alpha_base + alpha_g
    beta_total = beta_base + beta_g

    # 4. Aplicación de la fórmula: AGB = alpha * (H * CD)^beta * CF
    # Nota: El producto H * CD es la base de la potencia
    agb = alpha_total * (h * cd)**beta_total * cf

    return agb

In [7]:
def calculateAGB(path_las_trees, path_las_terrain, pred_pth):
    trees_points = read_las(path_las_trees)
    terrain_pts = read_las(path_las_terrain)

    preds = torch.load(pred_pth)

    # Crear array de etiquetas
    num_puntos = trees_points.shape[0]
    num_instancias = preds['pred_masks'].shape[0]
    labels = np.full(num_puntos, -1, dtype=np.int32)
    # IDs únicos a cada máscara
    for i in range(num_instancias):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        labels[mask] = i
    

    labels_terrain = np.full(terrain_pts.shape[0], -1, dtype=np.int32)
    
    labels = np.concatenate((labels, labels_terrain))

    puntos = np.vstack((trees_points, terrain_pts))

    z_suelo_puntos, altura_sobre_suelo, idx_x_all, idx_y_all, terrain_points = segment_terrain_points(puntos, resolution = 0.1)

    agbs = []

    for i in range(num_instancias):
        mask = labels == i
        puntos_arbol = puntos[mask]
        alturas_arbol = altura_sobre_suelo[mask]

        if len(puntos_arbol) < 10:
            continue
        # if i == 3:
        #     graficar_control_alpha(puntos_arbol, alpha=0.8)

        
        altura_maxima = np.max(alturas_arbol)
        diametro_copa, _ = calcular_diametro_equivalente_alpha(puntos_arbol, alpha=0.15)
        
        agb = calcular_agb(altura_maxima, diametro_copa)
        agbs.append(agb)

    return agbs



### Para watershed

In [17]:
import sys

root = Path("../..").resolve()
sys.path.append(str(root))

from scripts.treeSegWatershed import clasify_tree_watershed


In [22]:
import numpy as np

def calculateAGB_Watershed(arboles_df):
    # Obtenemos los IDs de las instancias (etiquetas)
    labels = arboles_df['label'].values
    # Identificamos IDs únicos ignorando el ruido (-1 o 0 según tu preprocesado)
    num_instancias = np.unique(labels[labels >= 0])

    agbs = []

    for i in num_instancias:
        # Filtramos el DataFrame directamente para mantener los nombres de columnas
        mask = labels == i
        datos_arbol = arboles_df[mask]
        
        if datos_arbol.empty:
            continue

        # 1. Extraer la Altura (H) en metros
        # Usamos el máximo de la columna 'Tree_Height'
        altura_maxima = datos_arbol['Tree_Height'].max()
        
        # 2. Extraer puntos para el Alpha Shape (necesita X e Y)
        # Convertimos a numpy solo lo necesario para la geometría
        puntos_geometria = datos_arbol[['X', 'Y']].values
        
        # 3. Calcular el Diámetro de Copa (CD) en metros
        # Tu función ya aplica la lógica de diámetro equivalente
        diametro_copa, _ = calcular_diametro_equivalente_alpha(puntos_geometria, alpha=0.15)
        
        # 4. Calcular AGB (kg) usando la fórmula de Jucker et al.
        # Gymnospermas: AGB = 0.109 * (H * CD)^1.790 * 1.2687
        agb = calcular_agb(altura_maxima, diametro_copa)
        agbs.append(agb)

    return agbs



In [49]:
#prueba
altura = 8.5    # metros
diametro = 4.2  # metros
biomasa = calcular_agb(altura, diametro)
print(f"Altura: {altura} m")
print(f"Diametro: {diametro} m")
print(f"Resultado del cálculo: {biomasa:.4f} Kg")

Altura: 8.5 m
Diametro: 4.2 m
Resultado del cálculo: 83.1929 Kg


## AGB para el conjunto de test

In [9]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

agbs_totales = []

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        terrain_path = os.path.join(base_path_terrain, las_name)
        agbs = calculateAGB(las_path, terrain_path, pred_path)
        agbs_totales.extend(agbs)

In [52]:
print(f"AGB promedio totales: {np.mean(agbs_totales):.4f} Kg por árbol")
print(f"AGB total estimada para el área: {np.sum(agbs_totales)/1000:.4f} Mg")

AGB promedio totales: 121.8052 Kg por árbol
AGB total estimada para el área: 8.7700 Mg


## AGB Final

In [23]:
import os
import re

# Ejecución
base_path_preds = "../../data/Santomera/result/FinalAGBResult"
base_path_gt1 = "../../data/Santomera/tiles/trees/labeled"
base_path_gt2 = "../../data/Santomera/tiles/trees/unlabeled"

base_path_terrain = "../../data/Santomera/tiles/terrain"

agbs_totales = []

regex = r"tile_\d+"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt1, las_name)
        if not os.path.exists(las_path):
            las_path = os.path.join(base_path_gt2, las_name)
        terrain_path = os.path.join(base_path_terrain, las_name)
        if las_path.endswith("_left.las") or las_path.endswith("_right.las"):
            name = re.search(regex, las_path).group(0)
            terrain_path = os.path.join(base_path_terrain, name) + ".las"
        agbs = calculateAGB(las_path, terrain_path, pred_path)
        agbs_totales.extend(agbs)

In [27]:
print(f"AGB total estimada para el área: {np.sum(agbs_totales)/1000:.4f} Mg")
print(f"AGB promedio totales: {np.mean(agbs_totales):.4f} Kg por árbol")
print(f"Desviación estándar de AGB: {np.std(agbs_totales):.4f} Kg por árbol")
print(f"Número total de árboles analizados: {len(agbs_totales)}")
print(f"AGB mínimo: {np.min(agbs_totales):.4f} Kg por árbol")
print(f"AGB máximo: {np.max(agbs_totales):.4f} Kg por árbol")

AGB total estimada para el área: 2058.1033 Mg
AGB promedio totales: 58.6488 Kg por árbol
Desviación estándar de AGB: 79.8373 Kg por árbol
Número total de árboles analizados: 35092
AGB mínimo: 0.0911 Kg por árbol
AGB máximo: 1636.3573 Kg por árbol


## Cálculo AGB para conference

### OACNN

In [8]:
import os

# Ejecución
base_path_preds_base = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    agbs_totales = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculateAGB(las_path, terrain_path, pred_path)
            agbs_totales.extend(agbs)
    print(f"Modelo OACNN{model}: AGB total: {np.sum(agbs_totales)/1000:.4f} Mg. Numero de árboles: {len(agbs_totales)}")

Modelo OACNN_0: AGB total: 13.7389 Mg. Numero de árboles: 211


Modelo OACNN_25: AGB total: 13.2509 Mg. Numero de árboles: 215


Modelo OACNN_50: AGB total: 12.9826 Mg. Numero de árboles: 180


Modelo OACNN_100: AGB total: 13.1812 Mg. Numero de árboles: 170


Modelo OACNN_ML: AGB total: 12.4703 Mg. Numero de árboles: 132


### PT-v3

In [9]:
import os

# Ejecución
base_path_preds_base = "../../data/Santomera/result/PT-v3"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    agbs_totales = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculateAGB(las_path, terrain_path, pred_path)
            agbs_totales.extend(agbs)
    print(f"Modelo PT-v3{model}: AGB total: {np.sum(agbs_totales)/1000:.4f} Mg. Numero de árboles: {len(agbs_totales)}")

Modelo PT-v3_0: AGB total: 12.5915 Mg. Numero de árboles: 117
Modelo PT-v3_25: AGB total: 12.5333 Mg. Numero de árboles: 122


Modelo PT-v3_50: AGB total: 12.5808 Mg. Numero de árboles: 144


Modelo PT-v3_100: AGB total: 12.2935 Mg. Numero de árboles: 103


Modelo PT-v3_ML: AGB total: 10.9493 Mg. Numero de árboles: 102


### SpUNET

In [10]:
import os

# Ejecución
base_path_preds_base = "../../data/Santomera/result/SpUNET"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
base_path_terrain = "../../data/Santomera/tiles/terrain"

models = ["_0", "_25", "_50", "_100", "_ML"]

for model in models:
    agbs_totales = []
    model_path = base_path_preds_base + model
    for file in os.listdir(model_path):
        if file.endswith(".pth"):
            pred_path = os.path.join(model_path, file)
            las_name = file.replace("_pred.pth", ".las")
            las_path = os.path.join(base_path_gt, las_name)
            terrain_path = os.path.join(base_path_terrain, las_name)
            agbs = calculateAGB(las_path, terrain_path, pred_path)
            agbs_totales.extend(agbs)
    print(f"Modelo SpUNET{model}: AGB total: {np.sum(agbs_totales)/1000:.4f} Mg. Numero de árboles: {len(agbs_totales)}")

Modelo SpUNET_0: AGB total: 8.6367 Mg. Numero de árboles: 200


Modelo SpUNET_25: AGB total: 9.8096 Mg. Numero de árboles: 322


Modelo SpUNET_50: AGB total: 12.1484 Mg. Numero de árboles: 180


Modelo SpUNET_100: AGB total: 12.4867 Mg. Numero de árboles: 163


Modelo SpUNET_ML: AGB total: 15.6206 Mg. Numero de árboles: 179


### Watershed

In [23]:

test_folder = "../../data/Santomera/tests"
agbs_totales = []
for file in os.listdir(test_folder):
    if file.endswith(".las"):
        ruta_trees = os.path.join(test_folder, file)
        ruta_terrain = os.path.join("../../data/Santomera/tiles/terrain", file)
        arboles_df = clasify_tree_watershed(ruta_trees, ruta_terrain, 0.5, 1.2, 0.05)
        if not arboles_df.empty:
            agbs = calculateAGB_Watershed(arboles_df)
            agbs_totales.extend(agbs)
        

print(f"AGB total estimada para el área: {np.sum(agbs_totales)/1000:.4f} Mg")
print(f"AGB promedio totales: {np.mean(agbs_totales):.4f} Kg por árbol")
print(f"Desviación estándar de AGB: {np.std(agbs_totales):.4f} Kg por árbol")
print(f"Número total de árboles analizados: {len(agbs_totales)}")

AGB total estimada para el área: 16.3456 Mg
AGB promedio totales: 259.4544 Kg por árbol
Desviación estándar de AGB: 281.8812 Kg por árbol
Número total de árboles analizados: 63
